# Experiment 7.3.3 — Affine Linear to LIF Substitution

Analysis-only notebook. It reads finalized Exp7.3.3 artifacts and does not train models or submit jobs.


In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def find_repo_root(start=Path.cwd()):
    cur = start.resolve()
    for candidate in (cur, *cur.parents):
        if (candidate / 'pyproject.toml').exists() and (candidate / 'scripts').exists():
            return candidate
    raise FileNotFoundError('Could not find repository root')

REPO_ROOT = find_repo_root()
ROOT = REPO_ROOT / 'notebooks' / 'artifacts' / 'experiment_7_3_3_affine_lif_substitution' / 'affine_lif_substitution_v1'
manifest = json.loads((ROOT / 'manifest.json').read_text())
runs = pd.read_csv(ROOT / 'method_runs.csv')
summary = pd.read_csv(ROOT / 'method_summary.csv')
contrasts = pd.read_csv(ROOT / 'contrast_summary.csv')
checks = pd.read_csv(ROOT / 'p7_reproduction_and_charge_checks.csv')
manifest


## P7 parameter reconstruction and beta=1 charge checks

Every row should show zero prediction mismatch and near-zero charge error.


In [ ]:
checks


## Test balanced accuracy

`analog_affine` is the recovered Exp7.3.2 P7 ceiling. All LIF/IF methods reuse exactly the same recovered W and effective sequence-level intercept; only the output dynamics and bias injection timing change.


In [ ]:
cols = [
    'backbone_objective', 'method',
    'test_ba_mean', 'test_ba_std',
    'source_p7_test_ba_mean', 'delta_vs_affine_p7_pp_mean',
]
summary[cols]


In [ ]:
method_order = manifest['methods']
for backbone in manifest['backbone_objectives']:
    view = summary[summary.backbone_objective == backbone].set_index('method').reindex(method_order)
    fig, ax = plt.subplots(figsize=(12, 5))
    x = np.arange(len(view))
    ax.bar(x, 100 * view.test_ba_mean.to_numpy(), yerr=100 * view.test_ba_std.fillna(0).to_numpy(), capsize=3)
    ax.axhline(100 * view.source_p7_test_ba_mean.iloc[0], linestyle='--', label='P7 affine reference')
    ax.set_xticks(x)
    ax.set_xticklabels(view.index, rotation=55, ha='right')
    ax.set_ylabel('Test balanced accuracy (%)')
    ax.set_title(f'Exp7.3.3 — {backbone} backbone')
    ax.legend()
    fig.tight_layout()
    plt.show()


## Mechanism contrasts

Key contrasts isolate removal of the affine offset, LIF realization loss, start/end one-shot bias benefit, and the beta=1 charge-preserving control.


In [ ]:
contrasts.sort_values(['backbone_objective', 'contrast'])
